# Sentiment & Emotion Classification v2

Model-agnostic pipeline for the 8 models in `huggingmodels.txt`:
- **Single-label** (sentiment / emotion): softmax → argmax → one label  
- **Multilabel** (emotion): sigmoid → threshold → list of active labels

**To switch models: edit Cell 2 only.**

Results are stored under `tweet['classifications'][MODEL_NAME]`, so multiple models can coexist in the same JSON file.

In [ ]:
%%time
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# Set to True if running on local machine with Google Drive Desktop mounted
# Set to False if running in Google Colab cloud
RUNNING_LOCALLY = False

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

twits_folder        = BASE_PATH / 'Raw Data/Twits/'
test_folder         = BASE_PATH / 'Raw Data/'
datasets_folder     = BASE_PATH / 'Data Sets'
cleanedds_folder    = BASE_PATH / 'Data Sets/Cleaned Data'
block_folder        = cleanedds_folder / 'Blocks'
networks_folder     = BASE_PATH / 'Data Sets/Networks/'
literature_folder   = BASE_PATH / 'Literature/'
topic_models_folder = BASE_PATH / 'Models/Topic Modeling/'

In [ ]:
%%time
# ─── ONLY THIS CELL NEEDS TO BE EDITED ────────────────────────────────────────

MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment-latest'

# Available models (see Cell 4 for full registry):
# 'cardiffnlp/twitter-roberta-base-sentiment-latest'          — sentiment, single-label (3 classes)
# 'cardiffnlp/twitter-xlm-roberta-base-sentiment'             — sentiment, single-label, multilingual
# 'cardiffnlp/twitter-roberta-base-emotion'                   — emotion,   single-label (4 classes)
# 'wesleyacheng/twitter-emotion-classification-with-bert'     — emotion,   single-label (3 classes)
# 'Supreeth/DeBERTa-Twitter-Emotion-Classification'           — emotion,   single-label (15 classes)
# 'cardiffnlp/twitter-roberta-base-emotion-latest'            — emotion,   multilabel  (11 classes)
# 'cardiffnlp/twitter-roberta-base-emotion-multilabel-latest' — emotion,   multilabel  (11 classes)
# 'bhadresh-savani/bert-base-go-emotion'                      — emotion,   multilabel  (28 classes)

MULTILABEL_THRESHOLD = 0.5   # score threshold for multilabel models
BATCH_SIZE           = 512   # tweets per GPU batch

# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
%%time
import json, os
from datetime import datetime
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
%%time
MODEL_REGISTRY = {
    'cardiffnlp/twitter-roberta-base-sentiment-latest': {
        'task': 'sentiment',
        'classification': 'single',
        'preprocessing': 'cardiffnlp',
        'labels': {0: 'negative', 1: 'neutral', 2: 'positive'},
    },
    'cardiffnlp/twitter-xlm-roberta-base-sentiment': {
        'task': 'sentiment',
        'classification': 'single',
        'preprocessing': 'cardiffnlp',
        'labels': {0: 'Negative', 1: 'Neutral', 2: 'Positive'},
    },
    'cardiffnlp/twitter-roberta-base-emotion': {
        'task': 'emotion',
        'classification': 'single',
        'preprocessing': 'cardiffnlp',
        'labels': {0: 'anger', 1: 'joy', 2: 'optimism', 3: 'sadness'},
    },
    'wesleyacheng/twitter-emotion-classification-with-bert': {
        'task': 'emotion',
        'classification': 'single',
        'preprocessing': 'none',
        'labels': {0: 'ANGER', 1: 'JOY', 2: 'SADNESS'},
    },
    'Supreeth/DeBERTa-Twitter-Emotion-Classification': {
        'task': 'emotion',
        'classification': 'single',
        'preprocessing': 'none',
        'labels': {
             0: 'anger',    1: 'boredom',  2: 'empty',   3: 'enthusiasm',
             4: 'fear',     5: 'fun',      6: 'happiness', 7: 'hate',
             8: 'joy',      9: 'love',    10: 'neutral',  11: 'relief',
            12: 'sadness', 13: 'surprise', 14: 'worry',
        },
    },
    'cardiffnlp/twitter-roberta-base-emotion-latest': {
        'task': 'emotion',
        'classification': 'multilabel',
        'preprocessing': 'cardiffnlp',
        'labels': {
            0: 'anger', 1: 'anticipation', 2: 'disgust',   3: 'fear',
            4: 'joy',   5: 'love',         6: 'optimism',  7: 'pessimism',
            8: 'sadness', 9: 'surprise',  10: 'trust',
        },
    },
    'cardiffnlp/twitter-roberta-base-emotion-multilabel-latest': {
        'task': 'emotion',
        'classification': 'multilabel',
        'preprocessing': 'cardiffnlp',
        'labels': {
            0: 'anger', 1: 'anticipation', 2: 'disgust',   3: 'fear',
            4: 'joy',   5: 'love',         6: 'optimism',  7: 'pessimism',
            8: 'sadness', 9: 'surprise',  10: 'trust',
        },
    },
    'bhadresh-savani/bert-base-go-emotion': {
        'task': 'emotion',
        'classification': 'multilabel',
        'preprocessing': 'none',
        'labels': {
             0: 'admiration',    1: 'amusement',     2: 'anger',       3: 'annoyance',
             4: 'approval',      5: 'caring',         6: 'confusion',   7: 'curiosity',
             8: 'desire',        9: 'disappointment', 10: 'disapproval', 11: 'disgust',
            12: 'embarrassment', 13: 'excitement',    14: 'fear',       15: 'gratitude',
            16: 'grief',         17: 'joy',           18: 'love',       19: 'nervousness',
            20: 'optimism',      21: 'pride',         22: 'realization', 23: 'relief',
            24: 'remorse',       25: 'sadness',       26: 'surprise',   27: 'neutral',
        },
    },
}

model_config = MODEL_REGISTRY[MODEL_NAME]
MODEL_ALIAS  = MODEL_NAME.split('/')[-1]

_task   = model_config['task']
_ctype  = model_config['classification']
_labels = list(model_config['labels'].values())
print(f'Model  : {MODEL_NAME}')
print(f'Task   : {_task}')
print(f'Type   : {_ctype}')
print(f'Labels ({len(_labels)}): {_labels}')

In [ ]:
%%time
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device).eval()

print('Model loaded and ready.')

In [ ]:
%%time
def preprocess(text, style='cardiffnlp'):
    '''Preprocess tweet text before tokenization.
    Cardiff NLP models expect @mentions replaced with @user and URLs with http.
    Other models use raw text.
    '''
    if style == 'cardiffnlp':
        tokens = []
        for t in text.split():
            t = '@user' if t.startswith('@') and len(t) > 1 else t
            t = 'http'  if t.startswith('http')               else t
            tokens.append(t)
        return ' '.join(tokens)
    return text


def run_batch(texts, model, tokenizer, model_config, threshold):
    '''Run one GPU batch. Returns a list of result dicts (one per tweet).
    Single-label  → {'label': str, 'scores': {label: float}}
    Multilabel    → {'labels': [str], 'scores': {label: float}}
    '''
    encoded = tokenizer(
        texts, padding=True, truncation=True, max_length=512, return_tensors='pt'
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        logits = model(**encoded).logits

    results = []
    if model_config['classification'] == 'single':
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        for prob in probs:
            scores = {model_config['labels'][j]: float(prob[j]) for j in range(len(prob))}
            best   = model_config['labels'][int(prob.argmax())]
            results.append({'label': best, 'scores': scores})
    else:  # multilabel — sigmoid, NOT softmax
        probs = torch.sigmoid(logits).cpu().numpy()
        for prob in probs:
            scores  = {model_config['labels'][j]: float(prob[j]) for j in range(len(prob))}
            actives = [model_config['labels'][j] for j in range(len(prob)) if prob[j] > threshold]
            results.append({'labels': actives, 'scores': scores})
    return results


def empty_result(model_config):
    '''Return a zeroed result for empty-text tweets.'''
    scores = {lbl: 0.0 for lbl in model_config['labels'].values()}
    if model_config['classification'] == 'single':
        fallback = list(model_config['labels'].values())[0]
        return {'label': fallback, 'scores': scores}
    return {'labels': [], 'scores': scores}

# Sanity Check Helper

In [ ]:
%%time
def sanity_check(output_path, model_name, model_config, num_samples=5):
    expected_keys = set(model_config['labels'].values())
    cls_type      = model_config['classification']
    print(f'Sanity check: {output_path}\n')

    with open(output_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            twit   = json.loads(line)
            result = twit.get('classifications', {}).get(model_name)

            if result is None:
                print(f'[Line {i+1}] MISSING classifications entry')
                continue

            scores     = result.get('scores', {})
            score_keys = set(scores.keys())
            if score_keys != expected_keys:
                print(f'[Line {i+1}] Key mismatch: {score_keys}')

            if cls_type == 'single':
                label = result.get('label', '?')
                print(f'[Line {i+1}] {label} | {scores}')
            else:
                labels = result.get('labels', [])
                top3   = dict(sorted(scores.items(), key=lambda x: -x[1])[:3])
                print(f'[Line {i+1}] {labels} | top-3: {top3}')

            if i + 1 >= num_samples:
                break

    print('\nSanity check complete.')

# Test Run (~881 tweets)

In [ ]:
%%time
input_path  = cleanedds_folder / 'AItrust_pruned_twits_test.json'
output_path = cleanedds_folder / f'AItrust_pruned_twits_test_classified_{MODEL_ALIAS}.json'

with open(input_path, 'r', encoding='utf-8') as f:
    total_lines = sum(1 for _ in f)

buffer, meta = [], []

with open(input_path, 'r', encoding='utf-8') as infile, \
     open(output_path, 'w', encoding='utf-8') as outfile:

    for line in tqdm(infile, total=total_lines, desc='Classifying test tweets'):
        twit = json.loads(line)
        if 'classifications' not in twit:
            twit['classifications'] = {}

        cleaned = preprocess(twit.get('text', ''), style=model_config['preprocessing'])

        if cleaned.strip():
            buffer.append(cleaned)
            meta.append(twit)
        else:
            twit['classifications'][MODEL_NAME] = empty_result(model_config)
            outfile.write(json.dumps(twit, ensure_ascii=False) + '\n')

        if len(buffer) >= BATCH_SIZE:
            for twit_i, result in zip(meta, run_batch(buffer, model, tokenizer, model_config, MULTILABEL_THRESHOLD)):
                twit_i['classifications'][MODEL_NAME] = result
                outfile.write(json.dumps(twit_i, ensure_ascii=False) + '\n')
            buffer.clear()
            meta.clear()

    if buffer:
        for twit_i, result in zip(meta, run_batch(buffer, model, tokenizer, model_config, MULTILABEL_THRESHOLD)):
            twit_i['classifications'][MODEL_NAME] = result
            outfile.write(json.dumps(twit_i, ensure_ascii=False) + '\n')

print(f'Saved to: {output_path}')

In [ ]:
%%time
sanity_check(output_path, MODEL_NAME, model_config)

# Full Dataset (~17.4M tweets)

Processed in 10 blocks. Each block is saved to `block_folder/MODEL_ALIAS/block_N.json`.
Blocks with ≥95% completion are skipped on re-run.
All blocks are merged into one final file at the end.

In [ ]:
%%time
input_path        = cleanedds_folder / 'AItrust_pruned_twits.json'
final_output_path = cleanedds_folder / f'AItrust_pruned_twits_classified_{MODEL_ALIAS}.json'

model_block_folder = block_folder / MODEL_ALIAS
model_block_folder.mkdir(parents=True, exist_ok=True)

NUM_BLOCKS           = 10
COMPLETION_THRESHOLD = 0.95


def count_lines(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return sum(1 for _ in f)
    except FileNotFoundError:
        return 0


print('Counting total tweets...')
total_tweets = count_lines(input_path)
print(f'Total tweets: {total_tweets:,}')

block_size      = total_tweets // NUM_BLOCKS
last_block_size = total_tweets - block_size * (NUM_BLOCKS - 1)

# ── Block loop ───────────────────────────────────────────────────────────────
for block_id in range(NUM_BLOCKS):
    block_path    = model_block_folder / f'block_{block_id}.json'
    expected_size = block_size if block_id < NUM_BLOCKS - 1 else last_block_size
    expected_min  = int(expected_size * COMPLETION_THRESHOLD)

    if block_path.exists():
        done = count_lines(block_path)
        if done >= expected_min:
            print(f'Block {block_id} already done ({done}/{expected_size}). Skipping.')
            continue
        print(f'Block {block_id} incomplete ({done}/{expected_size}). Reprocessing.')

    print(f'Processing block {block_id + 1}/{NUM_BLOCKS}...')
    start_line = block_id * block_size
    buffer, meta = [], []

    with open(input_path, 'r', encoding='utf-8') as infile, \
         open(block_path, 'w', encoding='utf-8') as outfile:

        for _ in range(start_line):
            try:
                next(infile)
            except StopIteration:
                break

        for _ in tqdm(range(expected_size), desc=f'Block {block_id}'):
            try:
                line = next(infile)
            except StopIteration:
                break

            twit = json.loads(line)
            if 'classifications' not in twit:
                twit['classifications'] = {}

            cleaned = preprocess(twit.get('text', ''), style=model_config['preprocessing'])

            if cleaned.strip():
                buffer.append(cleaned)
                meta.append(twit)
            else:
                twit['classifications'][MODEL_NAME] = empty_result(model_config)
                outfile.write(json.dumps(twit, ensure_ascii=False) + '\n')

            if len(buffer) >= BATCH_SIZE:
                for twit_i, result in zip(meta, run_batch(buffer, model, tokenizer, model_config, MULTILABEL_THRESHOLD)):
                    twit_i['classifications'][MODEL_NAME] = result
                    outfile.write(json.dumps(twit_i, ensure_ascii=False) + '\n')
                buffer.clear()
                meta.clear()

        if buffer:
            for twit_i, result in zip(meta, run_batch(buffer, model, tokenizer, model_config, MULTILABEL_THRESHOLD)):
                twit_i['classifications'][MODEL_NAME] = result
                outfile.write(json.dumps(twit_i, ensure_ascii=False) + '\n')

    print(f'Block {block_id} done.')

# ── Merge ────────────────────────────────────────────────────────────────────
print('Merging all blocks...')
with open(final_output_path, 'w', encoding='utf-8') as outfile:
    for block_id in range(NUM_BLOCKS):
        bp = model_block_folder / f'block_{block_id}.json'
        if not bp.exists():
            print(f'Block {block_id} missing. Skipping in merge.')
            continue
        with open(bp, 'r', encoding='utf-8') as infile:
            for line in infile:
                outfile.write(line)

print(f'Merge complete: {final_output_path}')

In [ ]:
%%time
sanity_check(final_output_path, MODEL_NAME, model_config)

# Visualizations

In [ ]:
%%time
def visualize_results(output_path, model_name, model_config, title_suffix=''):
    '''Plot classification results for any model in the registry.

    Single-label: count bar, pie (if ≤5 labels), avg confidence, proportion over time.
    Multilabel:   horizontal count bar, proportion-with-label over time.
    '''
    cls_type   = model_config['classification']
    all_labels = list(model_config['labels'].values())

    label_counts      = Counter()
    confidence_sums   = Counter()
    time_label_counts = {}   # date -> Counter(label -> count)
    tweet_per_day     = Counter()

    with open(output_path, 'r', encoding='utf-8') as f:
        for line in f:
            twit   = json.loads(line)
            result = twit.get('classifications', {}).get(model_name, {})
            date   = (twit.get('created_at') or '')[:10] or None

            if cls_type == 'single':
                label  = result.get('label')
                scores = result.get('scores', {})
                if label:
                    label_counts[label]    += 1
                    confidence_sums[label] += scores.get(label, 0.0)
                if date and label:
                    if date not in time_label_counts:
                        time_label_counts[date] = Counter()
                    time_label_counts[date][label] += 1
            else:
                actives = result.get('labels', [])
                for lbl in actives:
                    label_counts[lbl] += 1
                if date:
                    tweet_per_day[date] += 1
                    if date not in time_label_counts:
                        time_label_counts[date] = Counter()
                    for lbl in actives:
                        time_label_counts[date][lbl] += 1

    counts  = [label_counts.get(lbl, 0) for lbl in all_labels]
    use_pie = cls_type == 'single' and len(all_labels) <= 5
    n_plots = (4 if use_pie else 3) if cls_type == 'single' else 2
    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4))
    ax = 0

    # ── Count bar ──
    if cls_type == 'single':
        axes[ax].bar(all_labels, counts, color='steelblue')
        axes[ax].set_ylabel('Count')
        axes[ax].tick_params(axis='x', rotation=45)
        axes[ax].grid(axis='y', linestyle='--', alpha=0.7)
    else:
        axes[ax].barh(all_labels, counts, color='steelblue')
        axes[ax].set_xlabel('Count')
        axes[ax].grid(axis='x', linestyle='--', alpha=0.7)
    axes[ax].set_title(f'Label Counts {title_suffix}')
    ax += 1

    # ── Pie (single-label, ≤5 classes only) ──
    if use_pie:
        axes[ax].pie(counts, labels=all_labels, autopct='%1.1f%%', startangle=140)
        axes[ax].set_title(f'Distribution (%) {title_suffix}')
        ax += 1

    # ── Avg confidence (single-label only) ──
    if cls_type == 'single':
        avg_conf = [
            confidence_sums.get(lbl, 0.0) / max(label_counts.get(lbl, 1), 1)
            for lbl in all_labels
        ]
        axes[ax].bar(all_labels, avg_conf, color='orange')
        axes[ax].set_title(f'Avg Confidence {title_suffix}')
        axes[ax].set_ylim(0, 1)
        axes[ax].tick_params(axis='x', rotation=45)
        axes[ax].grid(axis='y', linestyle='--', alpha=0.7)
        ax += 1

    # ── Time series ──
    if time_label_counts:
        if cls_type == 'single':
            rows = [
                {'date': d, 'label': lbl, 'count': cnt}
                for d, day_counts in time_label_counts.items()
                for lbl, cnt in day_counts.items()
            ]
            val_col  = 'count'
            ts_title = 'Proportion Over Time'
        else:
            rows = [
                {'date': d, 'label': lbl,
                 'proportion': cnt / max(tweet_per_day[d], 1)}
                for d, day_counts in time_label_counts.items()
                for lbl, cnt in day_counts.items()
            ]
            val_col  = 'proportion'
            ts_title = 'Proportion with Label Over Time'

        if rows:
            df = pd.DataFrame(rows)
            df['date'] = pd.to_datetime(df['date'])
            pivot = df.pivot_table(
                index='date', columns='label', values=val_col, aggfunc='sum'
            ).fillna(0)
            if cls_type == 'single':
                pivot = pivot.div(pivot.sum(axis=1), axis=0)
            for col in pivot.columns:
                axes[ax].plot(pivot.index, pivot[col], label=col)
            axes[ax].set_title(f'{ts_title} {title_suffix}')
            axes[ax].set_xlabel('Date')
            axes[ax].set_ylabel('Proportion')
            axes[ax].legend(fontsize=6 if len(all_labels) > 5 else 8)
            axes[ax].grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

## For Test Data

In [ ]:
%%time
test_output = cleanedds_folder / f'AItrust_pruned_twits_test_classified_{MODEL_ALIAS}.json'
visualize_results(test_output, MODEL_NAME, model_config, title_suffix='(Test Data)')

## For Full Data

In [ ]:
%%time
full_output = cleanedds_folder / f'AItrust_pruned_twits_classified_{MODEL_ALIAS}.json'
visualize_results(full_output, MODEL_NAME, model_config, title_suffix='(Full Data)')

# Disconnect From Runtime

In [ ]:
%%time
from datetime import datetime
import pytz
from IPython.display import Javascript

nyc_time       = datetime.now(pytz.timezone('America/New_York'))
formatted_time = nyc_time.strftime('%Y-%m-%d %H:%M:%S %Z')
print(f'Disconnected from runtime at: {formatted_time}')

display(Javascript('google.colab.kernel.disconnect()'))